# Training Kraken BLLA (Fine-tuning) di Kaggle — dengan **Layer Freezing**, Checkpoint & Auto-Resume

Alur notebook:
1. Siapkan data: PAGE XML disalin ke `/kaggle/working/data/...` dengan `imageFilename` diarahkan ke gambar aslinya (karena folder `images/` dan `xml/` terpisah).
2. Kraken dipasang di **virtual environment terpisah** (sama seperti notebook baseline) agar tidak bentrok dengan paket Kaggle.
3. Training dijalankan lewat `ketos segtrain`. Setiap akhir epoch disimpan `last.ckpt` (plus 3 checkpoint terbaik) di `/kaggle/working/checkpoints/blla`.
4. **Setiap kali sel training dijalankan ulang, otomatis lanjut dari checkpoint terakhir** (di `/kaggle/working`, atau dari *notebook output* versi sebelumnya yang di-attach sebagai Input).

### Cara melanjutkan setelah session Kaggle habis/berhenti
- **Session masih hidup:** cukup jalankan ulang sel training → lanjut dari `last.ckpt`.
- **Session baru:** `Save Version` (Save & Run All, atau Quick Save) pada session lama supaya folder `checkpoints/` tersimpan sebagai output. Di session baru: *Add Input → Notebook Output → pilih notebook ini*, lalu *Run All*. Notebook akan menemukan `last.ckpt` di `/kaggle/input/...` dan melanjutkannya.

### Freezing layer (baru)
`ketos segtrain` **tidak punya opsi freeze** (opsi `--freeze-backbone` hanya ada di `ketos train` untuk recognition). Karena itu freeze dipasang lewat `train_wrapper.py`: sebuah *callback* Lightning disuntikkan ke `KrakenTrainer`, membekukan `N` layer pertama dari model BLLA saat training dimulai. Mode (atur di bagian konfigurasi):
- `none` — full finetune, tidak ada yang dibekukan.
- `encoder` — bekukan tumpukan konvolusi + GroupNorm (10 layer pertama), latih bagian LSTM dan output.
- `head_only` — bekukan semua kecuali layer output (paling ringan, paling cepat, kapasitas adaptasi paling kecil).
- `custom` — bekukan `FREEZE_CUSTOM_N` layer pertama.
- `UNFREEZE_AT_STEP` > 0 → bekukan dulu, lalu semua layer dibuka setelah sekian *optimizer step* (gradual unfreezing).

**Kenapa bukan LoRA/QLoRA?** BLLA hanya ~5 MB (konvolusi + LSTM kecil). LoRA/QLoRA dirancang untuk model raksasa (transformer berjuta-miliar parameter): pada model sekecil ini penghematan memorinya hampir nol, PEFT tidak mendukung layer LSTM, dan kuantisasi 4-bit (QLoRA) justru merusak akurasi tanpa manfaat. Selain itu bobot LoRA harus di-*merge* lagi agar bisa disimpan sebagai `.safetensors` kraken. Layer freezing adalah pilihan yang tepat untuk model ini. Checkpoint dan hasil akhirnya tetap berformat standar kraken, jadi bisa langsung dimuat di notebook evaluasi.

### Catatan konfigurasi (penting)
- **Batch size 4 × gradient accumulation 2:** training BLLA di Kraken 7.1.1 *hard-coded* `batch_size=1` per langkah (ukuran lebar halaman bervariasi, dan kode metrik validasi mengasumsikan batch 1) — tidak ada opsi `--batch-size` di `segtrain`. Agar hasilnya setara, dipakai `accumulate_grad_batches = 4 × 2 = 8` → **effective batch size = 8**, sama dengan konfigurasi yang diminta. Model ini memakai GroupNorm (bukan BatchNorm), jadi akumulasi gradien setara dengan batch besar.
- **Warm-up 200 steps** dihitung dalam *optimizer step* (setelah akumulasi), bukan per halaman.
- **Text direction `horizontal-lr`:** bukan parameter training di `segtrain`; ini opsi *inference* (`SegmentationInferenceConfig(text_direction="horizontal-lr")`, yang juga default). Dicatat di konfigurasi dan dipakai saat evaluasi.

Aktifkan *Internet* dan *GPU* (Settings).

## 0. Dependensi ringan + Konfigurasi  (⬅ ISI BAGIAN INI)

In [1]:
!pip install -q lxml tqdm huggingface_hub

In [2]:
import os, sys, json, glob, shutil, subprocess
import numpy as np

# ================== ISI BAGIAN INI ==================
DATASET_BASE = "/kaggle/input/datasets/gracians/greek-dataset-baseline"   # folder induk dataset
TRAIN_ROOT   = f"{DATASET_BASE}/train"      # berisi images/ dan xml/
VAL_ROOT     = f"{DATASET_BASE}/val"        # berisi images/ dan xml/ ; isi None -> split otomatis dari train (rasio PARTITION)
PARTITION    = 0.9                          # hanya dipakai bila VAL_ROOT = None

IMAGE_EXTS   = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
# ====================================================

# ---- Hyperparameter (sesuai permintaan) ----
OPTIMIZER      = "Adam"
LEARNING_RATE  = 0.001
BATCH_SIZE     = 4          # per "langkah logis"; lihat catatan: Kraken BLLA = 1 per forward, dikompensasi accumulation
GRAD_ACCUM     = 2
EPOCHS         = 50
SCHEDULER      = "cosine"
MIN_LR         = 0.00002
WARMUP_STEPS   = 200
WEIGHT_DECAY   = 0.00001
TEXT_DIRECTION = "horizontal-lr"    # dipakai saat inference/evaluasi (bukan opsi segtrain)

ACCUM_BATCHES  = BATCH_SIZE * GRAD_ACCUM    # = 8 -> effective batch size

# ---- Model awal ----
INIT_FROM_PRETRAINED = True    # True = fine-tune dari model HF (baseline); False = latih dari nol
HF_REPO_ID   = "small-models-for-glam/kraken-blla-base"
HF_FILENAME  = "blla.mlmodel"

# ---- Freezing layer (segtrain tidak punya opsi ini -> dipasang lewat train_wrapper.py) ----
FREEZE_MODE      = "encoder"   # "none" | "encoder" | "head_only" | "custom"
FREEZE_CUSTOM_N  = 5           # dipakai bila FREEZE_MODE = "custom": jumlah layer pertama yang dibekukan
UNFREEZE_AT_STEP = 0           # 0 = beku sampai selesai; >0 = buka semua layer setelah N optimizer step (gradual unfreeze)

_FREEZE_PRESETS = {"none": 0, "encoder": 10, "head_only": -1}   # angka negatif = bekukan semua KECUALI |n| layer terakhir
FREEZE_FIRST_N  = FREEZE_CUSTOM_N if FREEZE_MODE == "custom" else _FREEZE_PRESETS[FREEZE_MODE]
RUN_NAME        = f"freeze_{FREEZE_MODE}" + (f"_n{FREEZE_CUSTOM_N}" if FREEZE_MODE == "custom" else "") + (f"_unfreeze{UNFREEZE_AT_STEP}" if UNFREEZE_AT_STEP else "")

# ---- Lain-lain ----
SEED         = 42
NUM_WORKERS  = 2
PRECISION    = None            # None = default (32-true). Contoh lain: "16-mixed" (lebih cepat di T4, sedikit risiko tidak stabil)
SAVE_TOP_K   = 3               # jumlah checkpoint terbaik yang disimpan (selain last.ckpt) agar hemat disk

WORK_DIR     = "/kaggle/working"
CKPT_SUBDIR  = f"checkpoints/blla_{RUN_NAME}"   # folder terpisah per mode freeze agar tidak ter-resume dari run lain
CKPT_DIR     = f"{WORK_DIR}/{CKPT_SUBDIR}"          # <- checkpoint disimpan di sini (persist sebagai output Kaggle)
DATA_DIR     = f"{WORK_DIR}/data"                   # salinan XML dengan path gambar absolut
VENV_DIR     = "/tmp/kraken_env"

os.makedirs(CKPT_DIR, exist_ok=True)
print("Effective batch size:", ACCUM_BATCHES)
print("Mode freeze         :", FREEZE_MODE, f"(FREEZE_FIRST_N={FREEZE_FIRST_N}, UNFREEZE_AT_STEP={UNFREEZE_AT_STEP})")
print("Checkpoint dir      :", CKPT_DIR)

Effective batch size: 8
Mode freeze         : encoder (FREEZE_FIRST_N=10, UNFREEZE_AT_STEP=0)
Checkpoint dir      : /kaggle/working/checkpoints/blla_freeze_encoder


## 1. Siapkan data (PAGE XML + path gambar absolut)
Kraken membaca gambar dari atribut `imageFilename` relatif terhadap lokasi XML. Karena `images/` dan `xml/` terpisah (dan `/kaggle/input` read-only), XML disalin ke `/kaggle/working/data/` dengan `imageFilename` diganti path absolut. Langkah ini **idempotent** — dijalankan ulang di session baru akan menghasilkan path yang sama (penting agar resume dari checkpoint menemukan file datanya).

In [3]:
from lxml import etree

def ltag(e):
    return e.tag.split("}")[-1] if isinstance(e.tag, str) else ""

def index_files(folder, exts):
    d = {}
    for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True):
        if os.path.isfile(p) and p.lower().endswith(exts):
            d[os.path.splitext(os.path.basename(p))[0]] = p
    return d

def prepare_split(root, out_dir):
    imgs = index_files(f"{root}/images", IMAGE_EXTS)
    xmls = index_files(f"{root}/xml", (".xml",))
    os.makedirs(out_dir, exist_ok=True)
    ok, no_img, no_bl = [], [], []
    for stem, xp in sorted(xmls.items()):
        if stem not in imgs:
            no_img.append(stem); continue
        tree = etree.parse(xp)
        r = tree.getroot()
        page = next((e for e in r.iter() if ltag(e) == "Page"), None)
        if page is None:
            continue
        n_bl = sum(1 for e in r.iter() if ltag(e) == "Baseline" and e.get("points"))
        if n_bl == 0:                      # segtrain butuh baseline; halaman tanpa baseline dilewati
            no_bl.append(stem); continue
        page.set("imageFilename", os.path.abspath(imgs[stem]))
        out = os.path.join(out_dir, stem + ".xml")
        tree.write(out, xml_declaration=True, encoding="utf-8")
        ok.append(out)
    print(f"{root}\n   gambar={len(imgs)} xml={len(xmls)} -> dipakai={len(ok)} | tanpa gambar={len(no_img)} | tanpa Baseline={len(no_bl)}")
    if no_bl:
        print("   [PERINGATAN] XML tanpa <Baseline> dilewati (contoh):", no_bl[:5])
    return ok

train_files = prepare_split(TRAIN_ROOT, f"{DATA_DIR}/train")
if not train_files:
    raise SystemExit("Tidak ada data training valid. Periksa TRAIN_ROOT dan pastikan XML memiliki <Baseline>.")

val_files = []
if VAL_ROOT and os.path.isdir(VAL_ROOT):
    val_files = prepare_split(VAL_ROOT, f"{DATA_DIR}/val")
elif VAL_ROOT:
    print(f"[PERINGATAN] VAL_ROOT tidak ditemukan ({VAL_ROOT}) -> split otomatis {PARTITION:.0%}/{1-PARTITION:.0%} dari train.")

TRAIN_LIST, VAL_LIST = f"{DATA_DIR}/train_files.txt", f"{DATA_DIR}/val_files.txt"
open(TRAIN_LIST, "w").write("\n".join(train_files) + "\n")
if val_files:
    open(VAL_LIST, "w").write("\n".join(val_files) + "\n")
print("Train:", len(train_files), "| Val:", len(val_files) if val_files else f"auto-partition {1-PARTITION:.0%}")

/kaggle/input/datasets/gracians/greek-dataset-baseline/train
   gambar=206 xml=206 -> dipakai=206 | tanpa gambar=0 | tanpa Baseline=0
/kaggle/input/datasets/gracians/greek-dataset-baseline/val
   gambar=25 xml=25 -> dipakai=25 | tanpa gambar=0 | tanpa Baseline=0
Train: 206 | Val: 25


## 2. Pasang kraken di virtual environment terpisah
Dilewati otomatis bila venv sudah ada.

In [4]:
VENV_PY   = f"{VENV_DIR}/bin/python"
VENV_KETOS = f"{VENV_DIR}/bin/ketos"
CLEAN_ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONSTARTUP", "PYTHONHOME")}

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, env=CLEAN_ENV)

if not os.path.exists(VENV_PY):
    try:
        sh(f"{sys.executable} -m venv {VENV_DIR}")
    except subprocess.CalledProcessError:
        sh(f"{sys.executable} -m pip install -q virtualenv")
        shutil.rmtree(VENV_DIR, ignore_errors=True)
        sh(f"{sys.executable} -m virtualenv {VENV_DIR}")
    sh(f"{VENV_PY} -m pip install -q --no-cache-dir -U pip")
    sh(f"{VENV_PY} -m pip install -q --no-cache-dir 'kraken>=7.1.0' huggingface_hub pillow")
else:
    print("venv sudah ada, dilewati:", VENV_DIR)

sh(f"{VENV_PY} -m pip list 2>/dev/null | grep -i -E '^(kraken|torch|lightning) '")

r = subprocess.run([VENV_PY, "-c", "import torch; x=torch.zeros(1).cuda(); print(float((x+1).item()))"],
                   capture_output=True, text=True, env=CLEAN_ENV)
USE_GPU = (r.returncode == 0)
print("GPU dipakai:", USE_GPU, "" if USE_GPU else "(fallback ke CPU - training akan sangat lambat)")
if not USE_GPU:
    print("Alasan (baris terakhir):", (r.stderr.strip().splitlines() or ["tidak ada pesan"])[-1])

$ /usr/bin/python3 -m venv /tmp/kraken_env
$ /usr/bin/python3 -m pip install -q virtualenv


Error: Command '['/tmp/kraken_env/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.


$ /usr/bin/python3 -m virtualenv /tmp/kraken_env
created virtual environment CPython3.13.16.final.0-64-x86_64 in 407ms
  creator CPython3Posix(dest=/tmp/kraken_env, clear=False, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.3
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator
$ /tmp/kraken_env/bin/python -m pip install -q --no-cache-dir -U pip


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


$ /tmp/kraken_env/bin/python -m pip install -q --no-cache-dir 'kraken>=7.1.0' huggingface_hub pillow


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


$ /tmp/kraken_env/bin/python -m pip list 2>/dev/null | grep -i -E '^(kraken|torch|lightning) '
kraken                    7.1.1
lightning                 2.6.1
torch                     2.14.0
GPU dipakai: True 


## 3. Model awal (pretrained dari Hugging Face)

In [5]:
model_path = None
if INIT_FROM_PRETRAINED:
    from huggingface_hub import hf_hub_download
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_FILENAME, local_dir=f"{WORK_DIR}/kraken-blla-base")
    print("Model awal:", model_path)
else:
    print("Latih dari nol (INIT_FROM_PRETRAINED=False)")

blla.mlmodel: reconstructing file:   0%|          |  0.00B / 5.05MB            

blla.mlmodel: downloading bytes:           |  0.00B            

Model awal: /kaggle/working/kraken-blla-base/blla.mlmodel


## 4. Skrip pendukung & pencarian checkpoint terakhir
- `train_wrapper.py` menjalankan `ketos`, menyuntikkan callback **layer freezing** (lihat konfigurasi `FREEZE_*`), tetapi memaksa `ModelCheckpoint` menyimpan **`last.ckpt` tiap epoch** (bawaan Kraken hanya menyimpan checkpoint terbaik berdasarkan `val_metric`, sehingga epoch terakhir bisa tidak tersimpan) dan membatasi jumlah checkpoint terbaik (`SAVE_TOP_K`).
- Pencarian checkpoint: mengecek `/kaggle/working/checkpoints/blla` dan semua `/kaggle/input/**/checkpoints/blla` (notebook output versi sebelumnya); dipilih yang epoch-nya paling tinggi.

In [6]:
WRAPPER = r"""
import os
import lightning.pytorch.callbacks as _cbs
from lightning.pytorch.callbacks import Callback

# ---- 1) last.ckpt tiap epoch + batasi jumlah checkpoint terbaik ----
_Orig = _cbs.ModelCheckpoint
class _LastMC(_Orig):
    def __init__(self, *a, **kw):
        kw["save_last"] = True
        kw["save_top_k"] = int(os.environ.get("SAVE_TOP_K", "3"))
        super().__init__(*a, **kw)
_cbs.ModelCheckpoint = _LastMC      # segtrain mengimpor ModelCheckpoint saat dipanggil -> memakai versi ini

# ---- 2) Layer freezing (segtrain tidak punya opsi freeze) ----
FREEZE_FIRST_N = int(os.environ.get("FREEZE_FIRST_N", "0"))
UNFREEZE_AT_STEP = int(os.environ.get("UNFREEZE_AT_STEP", "0"))

def freeze_count(n_layers, n_first):
    k = n_first if n_first >= 0 else n_layers + n_first      # negatif = semua kecuali |n| layer terakhir
    return max(0, min(k, n_layers))

class LayerFreeze(Callback):
    def __init__(self, n_first, unfreeze_at):
        super().__init__()
        self.n_first, self.unfreeze_at, self.unfrozen = n_first, unfreeze_at, False

    def _layers(self, pl_module):
        return list(pl_module.net.nn.named_children())      # urutan layer sesuai spec VGSL

    def _set(self, layers, flag):
        for _, m in layers:
            for p in m.parameters():
                p.requires_grad_(flag)

    def on_train_start(self, trainer, pl_module):
        layers = self._layers(pl_module)
        k = freeze_count(len(layers), self.n_first)
        if self.unfreeze_at > 0 and trainer.global_step >= self.unfreeze_at:
            self.unfrozen = True                             # resume setelah titik unfreeze -> jangan bekukan lagi
            print(f"[FREEZE] global_step {trainer.global_step} >= {self.unfreeze_at}: semua layer trainable", flush=True)
            return
        self._set(layers[:k], False)
        tot = sum(p.numel() for p in pl_module.net.parameters())
        trn = sum(p.numel() for p in pl_module.net.parameters() if p.requires_grad)
        print(f"[FREEZE] {k}/{len(layers)} layer pertama dibekukan | parameter trainable {trn:,}/{tot:,}", flush=True)
        for i, (name, m) in enumerate(layers):
            n = sum(p.numel() for p in m.parameters())
            st = "beku " if i < k else "latih"
            print(f"   [{i:02d}] {st} {name:<12} {type(m).__name__:<22} {n:>10,} param", flush=True)

    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        if self.unfreeze_at > 0 and not self.unfrozen and trainer.global_step >= self.unfreeze_at:
            self._set(self._layers(pl_module), True)          # optimizer sudah memuat semua parameter -> cukup nyalakan grad
            self.unfrozen = True
            print(f"[FREEZE] step {trainer.global_step}: semua layer dibuka (unfreeze)", flush=True)

if FREEZE_FIRST_N != 0:
    import kraken.train as _kt
    _OrigTrainer = _kt.KrakenTrainer
    class _FreezeTrainer(_OrigTrainer):
        def __init__(self, *a, **kw):
            cbs = kw.get("callbacks", [])
            cbs = list(cbs) if isinstance(cbs, (list, tuple)) else [cbs]
            cbs.append(LayerFreeze(FREEZE_FIRST_N, UNFREEZE_AT_STEP))
            kw["callbacks"] = cbs
            super().__init__(*a, **kw)
    _kt.KrakenTrainer = _FreezeTrainer   # segtrain mengimpor KrakenTrainer saat dipanggil -> memakai versi ini

from kraken.ketos import cli
cli(prog_name="ketos")
"""
CKPT_INFO = r"""
import sys, json, torch
out = []
for p in sys.argv[1:]:
    try:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        out.append({"path": p, "epoch": int(ck.get("epoch", -1)), "global_step": int(ck.get("global_step", 0))})
    except Exception as e:
        out.append({"path": p, "error": str(e)[:200]})
print("JSON:" + json.dumps(out))
"""
open("/tmp/train_wrapper.py", "w").write(WRAPPER)
open("/tmp/ckpt_info.py", "w").write(CKPT_INFO)

def find_latest_checkpoint():
    cands = glob.glob(f"{CKPT_DIR}/*.ckpt")
    cands += glob.glob(f"/kaggle/input/**/{CKPT_SUBDIR}/*.ckpt", recursive=True)
    cands = sorted(set(cands))
    if not cands:
        return None, 0
    r = subprocess.run([VENV_PY, "/tmp/ckpt_info.py", *cands], capture_output=True, text=True, env=CLEAN_ENV)
    line = [l for l in r.stdout.splitlines() if l.startswith("JSON:")]
    if not line:
        print("Gagal membaca checkpoint:", r.stderr[-500:]); return None, 0
    infos = [i for i in json.loads(line[-1][5:]) if "error" not in i]
    for i in json.loads(line[-1][5:]):
        if "error" in i: print("  [lewati]", i["path"], i["error"])
    if not infos:
        return None, 0
    best = max(infos, key=lambda i: (i["epoch"], i["global_step"]))
    return best["path"], best["epoch"] + 1      # epoch di ckpt 0-based -> jumlah epoch selesai

resume_ckpt, completed = find_latest_checkpoint()
if resume_ckpt:
    print(f"Checkpoint terakhir : {resume_ckpt}\nEpoch selesai       : {completed}/{EPOCHS}")
else:
    print("Belum ada checkpoint -> training dimulai dari awal.")

Belum ada checkpoint -> training dimulai dari awal.


## 5. Jalankan training (otomatis resume)
Jalankan ulang sel ini kapan pun untuk melanjutkan. Bila sudah mencapai `EPOCHS`, sel ini tidak melatih lagi.

In [7]:
common = [
    "-o", CKPT_DIR,
    "-q", "fixed", "-N", str(EPOCHS), "-F", "1",
    "--optimizer", OPTIMIZER,
    "-r", str(LEARNING_RATE),
    "-w", str(WEIGHT_DECAY),
    "--schedule", SCHEDULER,
    "--cos-max", str(EPOCHS),            # cosine turun sampai MIN_LR tepat di epoch terakhir
    "--cos-min-lr", str(MIN_LR),
    "--warmup", str(WARMUP_STEPS),
    "--accumulate-grad-batches", str(ACCUM_BATCHES),
]
group = ["--seed", str(SEED), "-d", "cuda:0" if USE_GPU else "cpu", "--workers", str(NUM_WORKERS)]
if PRECISION:
    group += ["--precision", PRECISION]

if resume_ckpt:
    # Data & konfigurasi model dipulihkan dari checkpoint; hanya hyperparameter di atas yang di-override.
    seg_args = common + ["--resume", resume_ckpt]
else:
    seg_args = common + ["-f", "page", "-t", TRAIN_LIST, "--resize", "new"]
    if val_files:
        seg_args += ["-e", VAL_LIST]
    else:
        seg_args += ["-p", str(PARTITION)]
    if model_path:
        seg_args += ["-i", model_path]

cmd = [VENV_PY, "-u", "/tmp/train_wrapper.py", *group, "segtrain", *seg_args]
print("$", " ".join(cmd), "\n")

if resume_ckpt and completed >= EPOCHS:
    print(f"Training sudah selesai ({completed}/{EPOCHS} epoch). Tidak ada yang dilatih ulang.")
else:
    env = dict(CLEAN_ENV)
    env["SAVE_TOP_K"] = str(SAVE_TOP_K)
    env["FREEZE_FIRST_N"] = str(FREEZE_FIRST_N)
    env["UNFREEZE_AT_STEP"] = str(UNFREEZE_AT_STEP)
    if not USE_GPU:
        env["CUDA_VISIBLE_DEVICES"] = ""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        for line in proc.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        proc.terminate()
        print("\nDihentikan manual. Jalankan ulang sel ini untuk melanjutkan dari checkpoint terakhir.")
    rc = proc.wait()
    if rc != 0:
        print(f"\n[!] ketos keluar dengan kode {rc}. Jika ada 'checkpoint_abort.ckpt', jalankan ulang sel ini untuk melanjutkan.")

$ /tmp/kraken_env/bin/python -u /tmp/train_wrapper.py --seed 42 -d cuda:0 --workers 2 segtrain -o /kaggle/working/checkpoints/blla_freeze_encoder -q fixed -N 50 -F 1 --optimizer Adam -r 0.001 -w 1e-05 --schedule cosine --cos-max 50 --cos-min-lr 2e-05 --warmup 200 --accumulate-grad-batches 8 -f page -t /kaggle/working/data/train_files.txt --resize new -e /kaggle/working/data/val_files.txt -i /kaggle/working/kraken-blla-base/blla.mlmodel 

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
                  Training Class Summary                   
┏━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┓
┃ Category  ┃ Class   ┃ Label Index ┃ Merged With ┃ Count ┃
┡━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━┩
│ baselines │ default │           2 │             │  4094 │
│ regions   │ text    │           3 │             │   206 │
└───────────┴─────────┴─────────────┴─────────────┴───────┘
GPU available: True (cuda), used: Tr

## 6. Hasil: checkpoint & bobot terbaik

In [8]:
import time
print("Isi", CKPT_DIR)
for p in sorted(glob.glob(f"{CKPT_DIR}/*")):
    print(f"  {os.path.basename(p):<45} {os.path.getsize(p)/1e6:8.1f} MB   {time.ctime(os.path.getmtime(p))}")

best = sorted(glob.glob(f"{CKPT_DIR}/best_*.safetensors"))
if best:
    final = f"{WORK_DIR}/blla_finetuned_{RUN_NAME}_best.safetensors"
    shutil.copy(best[-1], final)
    print("\nBobot terbaik:", final)
    print("Untuk evaluasi: isi FT1_WEIGHTS (atau FT2_WEIGHTS) di notebook evaluasi dengan file ini (sudah berformat kraken lengkap -> dimuat langsung); inference memakai text_direction =", TEXT_DIRECTION)
else:
    print("\nBelum ada file best_*.safetensors (muncul otomatis setelah training selesai).")

Isi /kaggle/working/checkpoints/blla_freeze_encoder
  best_0.5316.safetensors                            5.1 MB   Sun Nov 15 13:44:02 2026
  checkpoint_19-0.5274.ckpt                          6.3 MB   Sun Nov 15 12:07:35 2026
  checkpoint_20-0.5316.ckpt                          6.3 MB   Sun Nov 15 12:10:49 2026
  checkpoint_23-0.5278.ckpt                          6.3 MB   Sun Nov 15 12:20:30 2026
  last.ckpt                                          6.3 MB   Sun Nov 15 12:20:31 2026

Bobot terbaik: /kaggle/working/blla_finetuned_freeze_encoder_best.safetensors
Untuk evaluasi: isi FT1_WEIGHTS (atau FT2_WEIGHTS) di notebook evaluasi dengan file ini (sudah berformat kraken lengkap -> dimuat langsung); inference memakai text_direction = horizontal-lr
